Cài đặt thư viện và môi trường


In [ ]:
!pip install -q paddlepaddle-gpu==2.6.1
!pip install -q paddleocr==2.7.3
!pip install -q "numpy<2.0.0"
!pip install -q Levenshtein pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires 

In [ ]:
!rm -rf local_data

In [ ]:
import os
import json
import unicodedata
import pandas as pd
import Levenshtein

from pathlib import Path
from tqdm import tqdm
from paddleocr import PaddleOCR
from google.colab import drive

drive.mount('/content/drive')

ZIP_PATH = Path('/content/drive/MyDrive/IntroToML - OCR - data/processed_data.zip')
LOCAL_ROOT = Path('/content/local_data')

if ZIP_PATH.exists():
    if not LOCAL_ROOT.exists():
        !unzip -q "{ZIP_PATH}" -d "{LOCAL_ROOT}"
    else:
        print("Dữ liệu đã có sẵn.")
else:
    print(f"❌ LỖI: Không tìm thấy file {ZIP_PATH}")

TEST_DIR = LOCAL_ROOT / 'test'

print(f"\n Đường dẫn thư mục Test: {TEST_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 Đường dẫn thư mục Test: /content/local_data/test


Các hàm chức năng

In [ ]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFC', text)
    return text.strip().lower()

def calculate_cer(pred: str, gt: str) -> float:
    pred = normalize_text(pred)
    gt = normalize_text(gt)

    # Nếu Ground Truth rỗng
    if len(gt) == 0:
        return 1.0 if len(pred) > 0 else 0.0

    # Tính số thao tác Thêm, Sửa, Xóa
    edit_dist = Levenshtein.distance(pred, gt)

    # CER = (Số lỗi) / (Tổng số ký tự của Ground Truth)
    cer = edit_dist / len(gt)
    return cer

Khởi tạo mô hình và vòng lặp suy luận

In [7]:
# Khởi tạo mô hình
print("Đang nạp mô hình PaddleOCR...")
ocr = PaddleOCR(use_angle_cls=False, lang='vi', show_log=False)

# Quét qua tất cả thư mục
all_test_samples = []
subfolders = [f for f in TEST_DIR.iterdir() if f.is_dir()]

for subfolder in subfolders:
    label_file = subfolder / 'label.json'
    if not label_file.exists():
        continue

    # Đọc file label.json
    with open(label_file, 'r', encoding='utf-8') as f:
        ground_truths = json.load(f)

    for img_name, gt_text in ground_truths.items():
        img_path = subfolder / img_name
        if img_path.exists():
            all_test_samples.append((img_path, gt_text))

print(f"Thực hiện đánh giá trên {len(all_test_samples)} ảnh \n")

# Vòng lặp đánh giá
total_cer = 0.0
error_logs = []

for img_path, gt_text in tqdm(all_test_samples, desc="Đang đánh giá"):
    result = ocr.ocr(str(img_path), det=False, cls=False)

    pred_text = ""
    confidence = 0.0

    if result and isinstance(result, list) and len(result[0]) > 0:
        pred_text = result[0][0][0]
        confidence = result[0][0][1]

    # Tính điểm
    cer_score = calculate_cer(pred_text, gt_text)
    total_cer += cer_score

    # Lưu lại ảnh lỗi kèm theo TÊN THƯ MỤC CON để dễ tìm file sau này
    if cer_score > 0:
        error_logs.append({
            "folder": img_path.parent.name, # Lấy tên thư mục chứa ảnh
            "image": img_path.name,
            "ground_truth": normalize_text(gt_text),
            "prediction": normalize_text(pred_text),
            "cer_score": round(cer_score, 4),
            "confidence": round(confidence, 4)
        })

# Tính toán CER trung bình
test_samples = len(all_test_samples)
average_cer = total_cer / test_samples if test_samples > 0 else 0

print(f"\nCER trung bình: {average_cer * 100:.2f} %")
print(f"Số lượng ảnh dự đoán sai: {len(error_logs)}")

Đang nạp mô hình PaddleOCR...
Thực hiện đánh giá trên 15000 ảnh 



Đang đánh giá: 100%|██████████| 15000/15000 [02:50<00:00, 88.17it/s]


CER trung bình: 57.29 %
Số lượng ảnh dự đoán sai: 13837


Xuất báo cáo

In [ ]:
# Chuyển đổi danh sách lỗi thành Pandas DataFrame
df_errors = pd.DataFrame(error_logs)

# Sắp xếp từ lỗi nặng nhất (CER cao) xuống lỗi nhẹ nhất
df_errors = df_errors.sort_values(by="cer_score", ascending=False)

# Lưu file báo cáo ra ngoài thư mục Dataset_Splitted để dễ tìm
report_path = LOCAL_ROOT / 'baseline_PPOCR_report.csv'
df_errors.to_csv(report_path, index=False, encoding='utf-8-sig')

drive_report_path = Path('/content/drive/MyDrive/IntroToML - OCR - data/baseline_PPOCR_report.csv')
!cp "{report_path}" "{drive_report_path}"

print(f"Đã lưu lại report của baseline_PPOCR")

Đã lưu lại report của baseline_PPOCR
